# Ep. 03 — LLM, context, limits, and sliding window

Content moved from Ep. 02, plus sliding window: simple call, forgetting, history, overflow, and sliding window.


## 1. Simple call

We make one LLM call and print the reply.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("NVIDIA_API_KEY", "").strip()
MODEL = os.environ.get("NVIDIA_MODEL", "meta/llama-3.1-8b-instruct")

def ask(messages):
    """messages: string OR list of {role, content} dicts."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Set NVIDIA_API_KEY and run again.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

reply = ask("Remember this secret: the codeword is ORBIT-77.")
print(reply)


## 2. No history

Now ask the model what it just said.


In [ ]:
# New call — do not pass the previous cell's history
reply2 = ask("What did you just say?")
print(reply2)


## 3. Build the history

To make the model "remember", use a list of `role` / `content` messages, latest at the end.


In [ ]:
history = [
    {"role": "user", "content": "Remember this secret: the codeword is ORBIT-77."},
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "What is the codeword?"},
]

reply3 = ask(history)
print(reply3)


## 4. Window overflow

If the history grows too large, the context window overflows and the provider returns an error.


In [ ]:
# Grow the history until the API fails (e.g. 400 context length exceeded)
big_history = [
    {"role": "user", "content": "Remember this secret: the codeword is ORBIT-77."},
    {"role": "assistant", "content": reply},
]

blob = "x" * 50_000  # large text only to fill the context

try:
    while True:
        big_history.append({"role": "user", "content": blob})
        big_history.append({"role": "assistant", "content": "ok"})
        print(f"messages in history: {len(big_history)}")
        ask(big_history)
except Exception as e:
    print("Provider error:")
    print(e)


## 5. Sliding window

When history fills the context window, the minimal mitigation is to **drop the oldest messages** and keep only the recent ones.

- The model **forgets the beginning** of the conversation.
- It keeps the **recent thread** and the call stops overflowing.
- This is not summarization: it is just a FIFO slice of the message list.

```text
[msg1][msg2][msg3][msg4][msg5][msg6]   ← overflows
              [msg3][msg4][msg5][msg6]   ← after sliding window (N=4)
```


In [ ]:
# Long history with the secret at the START and short turns at the end
history_sw = [
    {"role": "user", "content": "Remember this secret: the codeword is ORBIT-77."},
    {"role": "assistant", "content": reply},
]
for i in range(1, 9):
    history_sw.append({"role": "user", "content": f"Turn {i}: ok?"})
    history_sw.append({"role": "assistant", "content": f"ok {i}"})

N = 6  # keep only the N most recent messages
window = history_sw[-N:]
print(f"Original history: {len(history_sw)} msgs → window: {len(window)} msgs")
print("First msg in window:", window[0]["content"][:80])

out_sw = ask(window + [{"role": "user", "content": "What is the codeword?"}])
print(out_sw)
